# 02 - Build train / val / test splits

Constructs reproducible link-prediction splits on the PlantMetWiki heterogeneous graph and saves them to `data/processed/splits.pt`.

## Overview

### Goal
Construct reproducible, biologically meaningful train / val / test splits for link prediction on the PlantMetWiki heterogeneous graph, and characterise each split with statistics and visualisations.

---

### Prediction target
**`(Protein, catalyzes, Interaction)`** is a directly persisted edge on `heterodata.pt`. Its destination nodes are always `Conversion`-subtype `Interaction` nodes, i.e. reactions. See `01_explore_graph.ipynb` Sections 6-9 for the subtype analysis and the graph cleaning that produces this edge.

Its inverse `(Interaction, catalyzed_by, Protein)` is masked jointly in every split, so the model cannot infer target edges from their reverse.

---

### Splitting strategies

| # | Strategy | Description | When to use |
|---|----------|-------------|-------------|
| 1 | **Random (80/10/10)** | Edges shuffled and split uniformly at random. Negative samples are drawn at a 1:1 ratio. | Baseline; fastest to run; assumes i.i.d. edges. |
| 2 | **Taxa (organism-stratified)** | Entire organisms are held out for val/test. Edges shared between train and val/test organisms stay in train (leakage-free). Greedy filling keeps an ~80/10/10 edge balance. Built in `03_embeddings_fulldata.ipynb` §7.7-§7.8, because the held-out organisms are chosen by embedding coverage. | Tests generalisation across species, the realistic deployment scenario for a plant metabolomics tool. |
| 3 | **Pathway-balanced** | Whole pathways are removed for val/test, sampled proportionally to pathway size (Section 6 below). | Tests generalisation to unseen biological contexts. |

---

### Steps in this notebook

1. **Setup**: load `heterodata.pt` and set the random seed.
2. **Inspect edge types**: confirm the target and reverse edge counts.
3. **Random split**: apply `RandomLinkSplit` (PyG) and inspect the message-passing and label edges.
4. **Save random splits** to `data/processed/splits.pt`.
5. **Taxa split**: built in `03_embeddings_fulldata.ipynb` §7.7-§7.8 and saved to `data/processed/splits_taxa.pt`.
6. **Pathway-balanced split** (Section 6 below), independent of the embeddings.

---

### Metrics and diagnostics

| Metric | What it measures |
|--------|-----------------|
| **Edge counts per split** | Whether the 80/10/10 target is achieved in practice. |
| **Organism / pathway counts per split** | Coverage of biological entities across splits. |
| **Tanimoto similarity (organism pairs)** | Biochemical overlap between organisms; guides which ones to hold out. |
| **Cross-split Tanimoto (protein sets)** | Key diagnostic: low Train∩Val / Train∩Test (< 0.4) means the split is genuinely out-of-distribution. |
| **Excluded edges** | Target edges whose protein is multi-label or has only the "unknown organism" fallback taxon — always placed in train. |

## Splitting strategy

**Target edge type:** `(Protein, catalyzes, Interaction)`, the "enzyme catalyses reaction" relationship and the primary missing-link prediction target (11,980 edges).

**Random split protocol:**
- 80 / 10 / 10 train / val / test on edges of the target type
- `(Interaction, catalyzed_by, Protein)`, the semantic inverse, is masked jointly to prevent leakage
- Negative samples are drawn uniformly at random at a 1:1 ratio

**Output:** `data/processed/splits.pt`, a dict with keys `train`, `val` and `test` (each a `HeteroData`) plus metadata.

In [ ]:
from pathlib import Path
import torch
import pandas as pd
from torch_geometric.transforms import RandomLinkSplit

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

HETERO_PATH = DATA_DIR / "heterodata.pt"
SPLITS_PATH = DATA_DIR / "splits.pt"

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

print(f"Torch {torch.__version__}")
print(f"heterodata.pt: {'OK' if HETERO_PATH.exists() else 'MISSING – run scripts/build_pyg.py'}")

In [ ]:
data = torch.load(HETERO_PATH, weights_only=False)
print(data)
print()

# Summarise all edge types with counts to inform target selection
for et in data.edge_types:
    n = data[et].edge_index.shape[1]
    print(f"  {et[0]} -[{et[1]}]-> {et[2]} : {n:,}")

## 1. Choose prediction target

`(Protein, catalyzes, Interaction)` is a directly persisted edge on `heterodata.pt`. The reified `Catalysis` nodes were collapsed into it during graph cleaning (`01_explore_graph.ipynb` Section 9), after the cardinality check of Section 8.1 showed that every `Catalysis` node has exactly one `Protein` and one `Conversion` neighbour. Its destination nodes are always `Conversion`-subtype `Interaction` nodes: "this protein catalyses this reaction".

Its inverse `(Interaction, catalyzed_by, Protein)` is also a persisted edge, masked jointly in every split so the model cannot infer target edges from their reverse.

In [ ]:
# (Protein, catalyzes, Interaction) and its reverse (catalyzed_by) are persisted
# edges in heterodata.pt (see 01_explore_graph.ipynb Section 9), so they are
# read directly.
nodes = pd.read_csv(DATA_DIR / "nodes.tsv", sep="\t")

TARGET_EDGE = ("Protein", "catalyzes", "Interaction")
REV_EDGE    = ("Interaction", "catalyzed_by", "Protein")

n_target = data[TARGET_EDGE].edge_index.shape[1]
n_rev    = data[REV_EDGE].edge_index.shape[1]

print(f"Target  {TARGET_EDGE} (dst is always Conversion-subtype): {n_target:,} edges")
print(f"Inverse {REV_EDGE}: {n_rev:,} edges")
print()
print(f"Expected val  edges (positive): ~{int(n_target * 0.1):,}")
print(f"Expected test edges (positive): ~{int(n_target * 0.1):,}")

## 2. Apply RandomLinkSplit (80 / 10 / 10)

In [ ]:
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=False,
    add_negative_train_samples=True,
    neg_sampling_ratio=1.0,
    edge_types=[TARGET_EDGE],
    rev_edge_types=[REV_EDGE],
)

train_data, val_data, test_data = transform(data)
print("Split complete.")

## 3. Inspect split statistics

In [ ]:
def split_summary(name, split):
    ei = split[TARGET_EDGE].edge_label_index
    el = split[TARGET_EDGE].edge_label
    mp = split[TARGET_EDGE].edge_index        # edges kept for message passing
    n_pos = int((el == 1).sum())
    n_neg = int((el == 0).sum())
    print(f"{name:6s}  MP edges: {mp.shape[1]:6,}  |  labels: {n_pos:5,} pos  {n_neg:5,} neg")

split_summary("Train", train_data)
split_summary("Val  ", val_data)
split_summary("Test ", test_data)

total_target = data[TARGET_EDGE].edge_index.shape[1]
train_pos = int((train_data[TARGET_EDGE].edge_label == 1).sum())
val_pos   = int((val_data[TARGET_EDGE].edge_label   == 1).sum())
test_pos  = int((test_data[TARGET_EDGE].edge_label  == 1).sum())
print()
print(f"Positive edges: train {train_pos/total_target:.0%} | val {val_pos/total_target:.0%} | test {test_pos/total_target:.0%}")

## 4. Save splits

In [ ]:
torch.save(
    {
        "train": train_data,
        "val":   val_data,
        "test":  test_data,
        "target_edge": TARGET_EDGE,
        "rev_edge":    REV_EDGE,
        "split_params": {
            "num_val": 0.1,
            "num_test": 0.1,
            "neg_sampling_ratio": 1.0,
            "random_seed": RANDOM_SEED,
        },
    },
    SPLITS_PATH,
)
print(f"Saved -> {SPLITS_PATH}")

## 5. Organism-stratified (taxa) split

The taxa split is built in `03_embeddings_fulldata.ipynb` §7.7-§7.8 and saved to `data/processed/splits_taxa.pt`. Held-out organisms are chosen with a minimum embedding coverage, which is only known once the embeddings exist. Re-run that notebook to regenerate the file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FIG_DIR   = ROOT / "reports" / "figures"
TABLE_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Target edges and their pathway membership, used by the pathway-balanced
# split in Section 6.
prot_int = pd.DataFrame({
    "protein_idx":     data[TARGET_EDGE].edge_index[0].numpy(),
    "interaction_idx": data[TARGET_EDGE].edge_index[1].numpy(),
})
int_path = pd.DataFrame({
    "interaction_idx": data[("Interaction", "is_part_of", "Pathway")].edge_index[0].numpy(),
    "pathway_idx":     data[("Interaction", "is_part_of", "Pathway")].edge_index[1].numpy(),
})
n_total = len(prot_int)
print(f"Target edges available for splitting: {n_total:,}")

def set_tanimoto(a, b):
    i = len(a & b); u = len(a | b)
    return i / u if u else 0.0

def tanimoto_flag(t, lo=0.4, hi=0.6):
    if t < lo:  return "✓ low — good out-of-distribution signal"
    if t < hi:  return "⚠ moderate — acceptable but check organism choice"
    return            "✗ high — consider more taxonomically distant organisms"

## 6. Pathway-balanced split

Maps target edges to the pathways they belong to via  
`Protein → is_part_of → Interaction → is_part_of → Pathway`  
then holds out entire pathways for val/test.

**Scientific question this split answers:**  
*"Can the model predict missing enzyme–reaction links for pathway contexts it has never seen during training?"*  
This is complementary to the taxa split: instead of generalising across species, the model must generalise across biological processes (e.g., trained on glycolysis, evaluated on a novel secondary metabolism pathway).

In [ ]:
PATHWAY_SPLITS_PATH = DATA_DIR / "splits_pathway.pt"

# ── Pathway labels from nodes.tsv ──────────────────────────────────────────────
pathway_nodes = (
    nodes[nodes["node_type"] == "Pathway"]
    .reset_index(drop=True)
    .rename_axis("pathway_idx")
    .reset_index()
)
pathway_nodes["pathway_name"] = pathway_nodes["node_id"].str.rsplit("/", n=1).str[-1]

# ── Pathway → target-edge mapping ─────────────────────────────────────────────
path_to_targets = (
    prot_int
    .merge(int_path, on="interaction_idx")
    .drop_duplicates(["protein_idx", "interaction_idx", "pathway_idx"])
)

# Pathway size as number of distinct target (protein, interaction) pairs
pw_target_sizes = (
    path_to_targets.groupby("pathway_idx")
    .size().reset_index(name="n_target_edges")
)

# Pathway "steps" = distinct Interaction nodes (biochemical reactions)
pw_steps = (
    int_path.groupby("pathway_idx")["interaction_idx"]
    .nunique().reset_index(name="n_steps")
)

pathway_stats = (
    pw_target_sizes
    .merge(pw_steps, on="pathway_idx", how="outer")
    .merge(pathway_nodes[["pathway_idx", "pathway_name"]], on="pathway_idx", how="left")
)
pathway_stats["n_target_edges"] = pathway_stats["n_target_edges"].fillna(0).astype(int)
pathway_stats["n_steps"]        = pathway_stats["n_steps"].fillna(0).astype(int)
pathway_stats = pathway_stats.sort_values("n_target_edges", ascending=False).reset_index(drop=True)

# Target edges covered by at least one pathway
n_covered = path_to_targets[["protein_idx","interaction_idx"]].drop_duplicates().shape[0]
n_pw_with_targets = (pathway_stats["n_target_edges"] > 0).sum()

print(f"Total pathways                  : {len(pathway_stats):,}")
print(f"Pathways with ≥1 target edge    : {n_pw_with_targets:,}")
print(f"Target edges covered by pathways: {n_covered:,} / {n_total:,} ({100*n_covered/n_total:.1f}%)")
print()
print("Pathway size stats (target edges per pathway):")
sizes = pathway_stats[pathway_stats["n_target_edges"] > 0]["n_target_edges"]
print(f"  mean={sizes.mean():.1f}  median={sizes.median():.0f}  "
      f"max={sizes.max():,}  min={sizes.min()}")
print()
print("Top 15 pathways by target edge count:")
print(pathway_stats.head(15)[["pathway_name","n_target_edges","n_steps"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pw_pos = pathway_stats[pathway_stats["n_target_edges"] > 0]

# Left: size distribution
axes[0].hist(np.log1p(pw_pos["n_target_edges"]), bins=40, edgecolor="black", color="teal")
axes[0].set_xlabel("log(1 + target edges per pathway)")
axes[0].set_ylabel("Number of pathways")
axes[0].set_title("Pathway size distribution (target edges, log scale)")

# Right: top 20 pathways
top20 = pw_pos.head(20)
axes[1].barh(top20["pathway_name"][::-1], top20["n_target_edges"][::-1], color="teal")
axes[1].set_xlabel("Target edges  (Protein catalyzes Interaction)")
axes[1].set_title("Top 20 pathways by target edge count")

plt.tight_layout()
fig.savefig(FIG_DIR / "pathway_size_distribution.png")
plt.show()

pathway_stats.to_csv(TABLE_DIR / "pathway_sizes.tsv", sep="\t", index=False)

### 6.1 Sampling strategy — size-proportional random selection

#### Why not just pick the largest pathways?
Selecting only large pathways for val/test would bias evaluation towards well-annotated hub pathways (e.g., central carbon metabolism) and leave small, specialised pathways always in train. A model could score well by memorising large-pathway chemistry without learning anything about rare biosynthetic routes.

#### Size-proportional weighted shuffle
Pathways are assigned to a random permutation where each pathway's probability of appearing early is **proportional to its number of target edges**. We then greedily fill val until ≥ 10 % of total target edges are reached, then test for another ≥ 10 %. This guarantees:
- Larger pathways are more likely to be selected (so we reach the 10 % threshold without needing thousands of tiny pathways)
- The selection is still random — not deterministic top-N — so the split is representative of the full size distribution
- The random seed makes the split fully reproducible

#### Leakage handling — strict rule
An edge is assigned to val (or test) only if **none** of its pathways is a training pathway.

*Why strict here (unlike taxa)?*  
For the pathway split the scientific question is pathway-context generalisation. If a reaction appears in both glycolysis (train) and gluconeogenesis (val), the model was trained on that exact (Protein, Interaction) pair from the glycolysis pathway. Predicting it in gluconeogenesis is not a test of generalisation — it is recall. The strict rule ensures val/test edges are genuinely novel pathway contexts.

*Trade-off:* Interactions shared across many pathways will tend to stay in train, making val/test consist of pathway-exclusive reactions — which is precisely the biologically interesting held-out set. If the strict rule gives too few edges, the pathway size distribution (above) can guide a decision to relax it.

In [ ]:
# ── Size-proportional weighted shuffle ────────────────────────────────────────
pw_pos = pathway_stats[pathway_stats["n_target_edges"] > 0].copy()
n_pw   = len(pw_pos)

rng   = np.random.default_rng(RANDOM_SEED)
probs = pw_pos["n_target_edges"].values.astype(float)
probs /= probs.sum()

order      = rng.choice(n_pw, size=n_pw, replace=False, p=probs)
shuffled_pw = pw_pos.iloc[order].reset_index(drop=True)
cumsum      = shuffled_pw["n_target_edges"].cumsum()

val_cut  = int((cumsum <  0.10 * n_total).sum())
test_cut = int((cumsum <  0.20 * n_total).sum())

val_pathways   = set(shuffled_pw.iloc[:val_cut]["pathway_idx"])
test_pathways  = set(shuffled_pw.iloc[val_cut:test_cut]["pathway_idx"])
train_pathways = set(pw_pos["pathway_idx"]) - val_pathways - test_pathways

print(f"Train pathways : {len(train_pathways):4d}")
print(f"Val   pathways : {len(val_pathways):4d}")
print(f"Test  pathways : {len(test_pathways):4d}")
print()

# ── Strict edge assignment ─────────────────────────────────────────────────────
# Edge → val/test only if NONE of its pathways is a training pathway
edge_pw_map = (
    path_to_targets
    .groupby(["protein_idx", "interaction_idx"])["pathway_idx"]
    .apply(frozenset).reset_index(name="pathways")
)

def assign_split_pw(pathways):
    has_train = bool(pathways & train_pathways)
    has_val   = bool(pathways & val_pathways)
    has_test  = bool(pathways & test_pathways)
    if   has_val  and not has_train: return "val"
    elif has_test and not has_train: return "test"
    else:                            return "train"

edge_pw_map["split"] = edge_pw_map["pathways"].apply(assign_split_pw)

# Edges in no pathway → train
no_pw_mask = ~prot_int.set_index(["protein_idx","interaction_idx"]).index.isin(
    edge_pw_map.set_index(["protein_idx","interaction_idx"]).index
)
no_pw_df          = prot_int[no_pw_mask].copy()
no_pw_df["pathways"] = [frozenset()] * len(no_pw_df)
no_pw_df["split"]    = "train"

all_edges_pw = pd.concat([edge_pw_map, no_pw_df], ignore_index=True)

sc_pw = all_edges_pw["split"].value_counts()
print("Edge assignment:")
for s in ["train", "val", "test"]:
    print(f"  {s:5s}: {sc_pw.get(s,0):6,}  ({100*sc_pw.get(s,0)/n_total:.1f}%)")

In [ ]:
# ── Cross-split metrics ────────────────────────────────────────────────────────
def split_pw_protein_set(split_name):
    return set(all_edges_pw[all_edges_pw["split"] == split_name]["protein_idx"].values)

def split_pw_interaction_set(split_name):
    return set(all_edges_pw[all_edges_pw["split"] == split_name]["interaction_idx"].values)

train_pw_prots = split_pw_protein_set("train")
val_pw_prots   = split_pw_protein_set("val")
test_pw_prots  = split_pw_protein_set("test")

tan_pw_tv = set_tanimoto(train_pw_prots, val_pw_prots)
tan_pw_tt = set_tanimoto(train_pw_prots, test_pw_prots)
tan_pw_vt = set_tanimoto(val_pw_prots,   test_pw_prots)

train_pw_ints = split_pw_interaction_set("train")
val_pw_ints   = split_pw_interaction_set("val")
test_pw_ints  = split_pw_interaction_set("test")

tan_int_tv = set_tanimoto(train_pw_ints, val_pw_ints)
tan_int_tt = set_tanimoto(train_pw_ints, test_pw_ints)
tan_int_vt = set_tanimoto(val_pw_ints,   test_pw_ints)

# ── Size distribution of val/test pathways ─────────────────────────────────────
val_pw_stats  = pathway_stats[pathway_stats["pathway_idx"].isin(val_pathways)]
test_pw_stats = pathway_stats[pathway_stats["pathway_idx"].isin(test_pathways)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Edge counts
splits_pw = ["train", "val", "test"]
colors_pw = ["steelblue", "darkorange", "seagreen"]
axes[0].bar(splits_pw, [sc_pw.get(s, 0) for s in splits_pw], color=colors_pw)
for i, s in enumerate(splits_pw):
    axes[0].text(i, sc_pw.get(s, 0) + 50, f"{sc_pw.get(s,0):,}", ha="center", fontsize=9)
axes[0].set_ylabel("Target edges (positives)")
axes[0].set_title("Pathway split — edge counts")

# 2. Pathway size distributions (val vs test)
axes[1].hist(np.log1p(val_pw_stats["n_target_edges"]),  bins=20, alpha=0.6,
             label=f"Val ({len(val_pathways)} pathways)",  color="darkorange", edgecolor="black")
axes[1].hist(np.log1p(test_pw_stats["n_target_edges"]), bins=20, alpha=0.6,
             label=f"Test ({len(test_pathways)} pathways)", color="seagreen",    edgecolor="black")
axes[1].set_xlabel("log(1 + target edges per pathway)")
axes[1].set_ylabel("Number of pathways")
axes[1].set_title("Size distribution of held-out pathways")
axes[1].legend()

# 3. Cross-split Tanimoto (protein sets)
tan_labels = ["Train∩Val", "Train∩Test", "Val∩Test"]
tan_vals   = [tan_pw_tv, tan_pw_tt, tan_pw_vt]
axes[2].bar(tan_labels, tan_vals, color=["mediumpurple", "teal", "salmon"])
axes[2].set_ylim(0, 1)
axes[2].set_ylabel("Tanimoto similarity")
axes[2].set_title("Cross-split protein overlap")
for i, v in enumerate(tan_vals):
    axes[2].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

plt.tight_layout()
fig.savefig(FIG_DIR / "pathway_split_stats.png")
plt.show()

print("Cross-split Tanimoto — protein sets:")
print(f"  Train vs Val : {tan_pw_tv:.4f}")
print(f"  Train vs Test: {tan_pw_tt:.4f}")
print(f"  Val vs Test  : {tan_pw_vt:.4f}")
print()
print("Cross-split Tanimoto — interaction sets:")
print(f"  Train vs Val : {tan_int_tv:.4f}")
print(f"  Train vs Test: {tan_int_tt:.4f}")
print(f"  Val vs Test  : {tan_int_vt:.4f}")

In [ ]:
from IPython.display import Markdown

# ── Save ───────────────────────────────────────────────────────────────────────
def make_pw_edge_index(split_name):
    sub = all_edges_pw[all_edges_pw["split"] == split_name]
    return torch.stack([
        torch.tensor(sub["protein_idx"].values,     dtype=torch.long),
        torch.tensor(sub["interaction_idx"].values, dtype=torch.long),
    ], dim=0)

train_pw_ei = make_pw_edge_index("train")
val_pw_ei   = make_pw_edge_index("val")
test_pw_ei  = make_pw_edge_index("test")

torch.save(
    {
        "train_edge_index": train_pw_ei,
        "val_edge_index":   val_pw_ei,
        "test_edge_index":  test_pw_ei,
        "target_edge":      TARGET_EDGE,
        "train_pathways":   sorted(train_pathways),
        "val_pathways":     sorted(val_pathways),
        "test_pathways":    sorted(test_pathways),
        "split_params": {
            "strategy":          "pathway_size_proportional",
            "leakage_rule":      "strict",
            "val_target_frac":   0.10,
            "test_target_frac":  0.10,
            "random_seed":       RANDOM_SEED,
        },
        "tanimoto": {
            "proteins": {"train_val": float(tan_pw_tv), "train_test": float(tan_pw_tt), "val_test": float(tan_pw_vt)},
            "interactions": {"train_val": float(tan_int_tv), "train_test": float(tan_int_tt), "val_test": float(tan_int_vt)},
        },
    },
    PATHWAY_SPLITS_PATH,
)
print(f"Saved → {PATHWAY_SPLITS_PATH}")
print(f"  train: {train_pw_ei.shape[1]:,}  val: {val_pw_ei.shape[1]:,}  test: {test_pw_ei.shape[1]:,}")

# ── Summary report ─────────────────────────────────────────────────────────────
n_train_pw = sc_pw.get("train", 0)
n_val_pw   = sc_pw.get("val",   0)
n_test_pw  = sc_pw.get("test",  0)

n_unassigned_pw = len(no_pw_df)
pct_covered     = 100 * n_covered / n_total

display(Markdown(f"""
### Pathway split — final results

#### Coverage

{n_pw_with_targets:,} of {len(pathway_stats):,} pathways contain at least one target edge, covering
{n_covered:,} / {n_total:,} ({pct_covered:.1f} %) target edges.
The remaining {n_total - n_covered:,} edges ({100*(n_total-n_covered)/n_total:.1f} %) belong to Interactions
not linked to any pathway — these are assigned to train.

Pathway size is highly skewed (mean = {sizes.mean():.1f}, median = {sizes.median():.0f} target edges),
confirming that size-proportional sampling is necessary to avoid over-representing hub pathways.

---

#### Split composition  (target: 80/10/10)

| Split  | Edges   | % total | Pathways held out |
|--------|--------:|--------:|:-----------------:|
| Train  | {n_train_pw:,} | {100*n_train_pw/n_total:.1f} % | {len(train_pathways)} |
| Val    | {n_val_pw:,}   | {100*n_val_pw/n_total:.1f} %   | {len(val_pathways)} |
| Test   | {n_test_pw:,}  | {100*n_test_pw/n_total:.1f} %  | {len(test_pathways)} |

Achieved split: **{100*n_train_pw/n_total:.0f}/{100*n_val_pw/n_total:.0f}/{100*n_test_pw/n_total:.0f}**.

---

#### Leakage handling — strict rule

An edge is assigned to val/test only if **none** of its pathways is a training pathway.
This ensures val/test evaluate exclusively on *pathway-context generalisation*: reactions
the model has never encountered in any pathway context.
Edges shared between training and held-out pathways stay in train.

---

#### Biochemical diversity — cross-split Tanimoto

**Protein sets** (how many enzymes are shared between splits):

| Pair | Tanimoto | Assessment |
|------|:--------:|-----------|
| Train ∩ Val  | {tan_pw_tv:.4f} | {tanimoto_flag(tan_pw_tv)} |
| Train ∩ Test | {tan_pw_tt:.4f} | {tanimoto_flag(tan_pw_tt)} |
| Val ∩ Test   | {tan_pw_vt:.4f} | {tanimoto_flag(tan_pw_vt)} |

**Interaction sets** (how many reactions are shared — pathway-specific diagnostic):

| Pair | Tanimoto |
|------|:--------:|
| Train ∩ Val  | {tan_int_tv:.4f} |
| Train ∩ Test | {tan_int_tt:.4f} |
| Val ∩ Test   | {tan_int_vt:.4f} |

The interaction Tanimoto is expected to be near 0 by construction (strict rule ensures
no interaction appears in both a training pathway and a val/test pathway exclusively).
The protein Tanimoto may be higher if the same enzyme appears across many pathway contexts.

---

#### Files saved
- `data/processed/splits_pathway.pt` — edge indices, pathway assignments, Tanimoto scores, split parameters.
"""))